In [ ]:
!pip install -q \
faiss-cpu \
sentence-transformers \
transformers \
accelerate \
torch


In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

SAVE_PATH = "../index/"

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

embed_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device=device
)

index = faiss.read_index(SAVE_PATH + "index.faiss")
chunks = np.load(SAVE_PATH + "chunks.npy", allow_pickle=True)
metadata = np.load(SAVE_PATH + "metadata.npy", allow_pickle=True)

print("✅ IITP knowledge loaded")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    temperature=0.1,
    return_full_text=False,
    eos_token_id=tokenizer.eos_token_id
)




In [ ]:
def ask(query, k=3):

    q_emb = embed_model.encode([query])
    D, I = index.search(q_emb, k)

    context = "\n".join([chunks[i] for i in I[0]])

    prompt = f"""
You are IIT Patna Academic Assistant.

Answer ONLY the question in 2-3 lines.
Do not repeat the context.
Stop after giving the answer.

Context:
{context}

Question:
{query}

Answer:
"""

    output = llm(prompt)


    answer = output[0]["generated_text"].strip()

    return answer.split("Question:")[0].strip()


In [ ]:
# question = "Where is IIT Patna located?"
# question = "When was IIT Patna started?"
question = "Name some features of IIT Patna"
#question = "What is eligibility for MTech admission at IIT Patna?"
#question = "Who is Prof. Rajiv Misra in IIT Patna?"
#question = "When is the next holiday as per the Academic Calander of IIT Patna?"

text = ask(question)
print (f"\nAnswer: ----------------------\n{text}")